# Volatility Assessment Engine - Configuration Demo

This notebook demonstrates the configurable volatility assessment engine that extracts parameters from the original oracle_integration.py and makes them configurable via YAML.

## Features Demonstrated
- YAML-based configuration loading
- Configurable price staleness scoring
- Configurable volatility calculation parameters
- Configurable manipulation detection thresholds
- Oracle confidence scoring with configurable weights
- MCP service integration (ports 8002/8003)

## Prerequisites
- MCP services running on ports 8002 (Algorand reader) and 8003 (Market data)
- Python packages: yaml, aiohttp, statistics

In [ ]:
# Setup imports and path
import sys
import os
import asyncio
from pathlib import Path
from datetime import datetime, timedelta
import json

# Add parent directory to path
current_dir = Path().absolute()
parent_dir = current_dir.parent
sys.path.insert(0, str(parent_dir))

# Import our modules
from core.volatility_engine import VolatilityAssessmentEngine, PriceOracle
from core.config import load_config

print("✓ Imports successful")
print(f"Working directory: {current_dir}")
print(f"Parent directory: {parent_dir}")

## 1. Configuration Loading and Inspection

Let's load the configuration and examine the parameters that were extracted from the original oracle_integration.py code.

In [ ]:
# Load configuration
config = load_config()

print("=== MCP Service Configuration ===")
print(f"Algorand Reader URL: {config.mcp_services.algorand_reader_url}")
print(f"Market Data URL: {config.mcp_services.market_data_url}")

print("\n=== Price Staleness Configuration ===")
staleness = config.price_staleness
print("Scoring Thresholds (seconds):")
for key, value in staleness.scoring_thresholds.items():
    print(f"  {key}: {value}s")

print("\nScoring Values:")
for key, value in staleness.scoring_values.items():
    print(f"  {key}: {value}")

print("\n=== Volatility Calculation Configuration ===")
vol_config = config.volatility_calculation
print("Lookback Periods:")
for key, value in vol_config.lookback_periods.items():
    print(f"  {key}: {value}")

print("\nWindows:")
for key, value in vol_config.windows.items():
    print(f"  {key}: {value}")

print(f"\nMinimum Data Points: {vol_config.minimum_data_points}")

In [ ]:
print("=== Manipulation Detection Configuration ===")
manip_config = config.manipulation_detection

print("Risk Thresholds:")
for key, value in manip_config.risk_thresholds.items():
    print(f"  {key}: {value}")

print("\nRisk Scoring Weights:")
for key, value in manip_config.risk_scoring.items():
    print(f"  {key}: {value}")

print("\nRisk Level Thresholds:")
for key, value in manip_config.risk_levels.items():
    print(f"  {key}: {value}")

print("\n=== Oracle Confidence Configuration ===")
oracle_config = config.oracle_confidence
print(f"Minimum Confidence: {oracle_config.minimum_confidence}")
print(f"Single Oracle Penalty: {oracle_config.confidence_penalty_single}")
print(f"Single Oracle Consensus: {oracle_config.consensus_strength_single}")

print("\nReliability Weights:")
for key, value in oracle_config.reliability_weights.items():
    print(f"  {key}: {value}")

## 2. Engine Initialization and Oracle Setup

Initialize the volatility assessment engine with our configuration and set up demo oracles.

In [ ]:
# Initialize the engine
engine = VolatilityAssessmentEngine()

# Setup demo oracles
engine.setup_demo_oracles()

print("=== Registered Oracles ===")
for oracle_name, oracle in engine.oracles.items():
    reliability = engine.reliability_scores[oracle_name]
    status = engine.oracle_status[oracle_name]
    print(f"  {oracle_name}: reliability={reliability:.3f}, status={status.value}")

print(f"\nTotal oracles registered: {len(engine.oracles)}")

## 3. Price Feed Retrieval and Analysis

Test the price feed retrieval functionality with multiple assets.

In [ ]:
# Test price feeds for multiple assets
test_assets = ["ALGO", "USDC", "BTC", "ETH"]

print("=== Price Feed Retrieval ===")
for asset in test_assets:
    feeds = engine.get_price_feeds(asset)
    print(f"\n{asset}:")
    
    if feeds:
        for feed in feeds:
            age_seconds = (datetime.now() - feed.timestamp).total_seconds()
            print(f"  {feed.oracle_name}: ${feed.price_usd:.4f} "
                  f"(confidence: {feed.confidence:.3f}, age: {age_seconds:.1f}s)")
    else:
        print(f"  No feeds available for {asset}")

## 4. Price Aggregation with Configurable Parameters

Demonstrate price aggregation using the configurable staleness scoring and reliability weights.

In [ ]:
# Test price aggregation for ALGO
asset_symbol = "ALGO"
feeds = engine.get_price_feeds(asset_symbol)

if feeds:
    print(f"=== Price Aggregation for {asset_symbol} ===")
    print(f"Raw feeds: {len(feeds)}")
    
    # Show individual feeds
    for i, feed in enumerate(feeds, 1):
        print(f"  Feed {i} ({feed.oracle_name}): ${feed.price_usd:.4f}")
    
    # Calculate staleness score using configuration
    staleness_score = engine.calculate_staleness_score(feeds)
    print(f"\nStaleness Score (configured): {staleness_score:.3f}")
    
    # Aggregate prices
    aggregated = engine.aggregate_prices(feeds)
    
    print(f"\n=== Aggregated Results ===")
    print(f"Consensus Price: ${aggregated.consensus_price:.4f}")
    print(f"Price Confidence: {aggregated.price_confidence:.3f}")
    print(f"Price Deviation: {aggregated.price_deviation:.4f}")
    print(f"Oracle Count: {aggregated.oracle_count}")
    print(f"Staleness Score: {aggregated.staleness_score:.3f}")
    print(f"Reliability Score: {aggregated.reliability_score:.3f}")
    print(f"Consensus Strength: {aggregated.consensus_strength:.3f}")
    
else:
    print(f"No price feeds available for {asset_symbol}")

## 5. Manipulation Detection with Configurable Thresholds

Test the price manipulation detection using the configurable thresholds and risk scoring.

In [ ]:
# Simulate price history for manipulation detection
print("=== Simulating Price History ===")
engine.simulate_price_history("ALGO", hours=2)

if "ALGO" in engine.price_history:
    history_count = len(engine.price_history["ALGO"])
    print(f"Generated {history_count} historical price points")
    
    # Show sample of recent history
    recent_feeds = sorted(engine.price_history["ALGO"], key=lambda x: x.timestamp, reverse=True)[:5]
    print("\nRecent price history (last 5 points):")
    for feed in recent_feeds:
        print(f"  {feed.timestamp.strftime('%H:%M:%S')} - {feed.oracle_name}: ${feed.price_usd:.4f}")
else:
    print("Failed to generate price history")

In [ ]:
# Test manipulation detection
print("=== Manipulation Detection Analysis ===")

# Test with different lookback periods
lookback_periods = [30, 60, 120]  # minutes

for lookback in lookback_periods:
    results = engine.detect_price_manipulation("ALGO", lookback_minutes=lookback)
    
    print(f"\nLookback: {lookback} minutes")
    print(f"  Risk Level: {results['manipulation_risk']}")
    print(f"  Risk Score: {results['risk_score']:.3f}")
    print(f"  Confidence: {results['confidence']:.3f}")
    print(f"  Price Volatility: {results['price_volatility']:.4f}")
    print(f"  Oracle Consensus: {results['oracle_consensus']:.3f}")
    print(f"  Recommendation: {results['recommendation']}")
    
    if results['risk_factors']:
        print(f"  Risk Factors: {', '.join(results['risk_factors'])}")
    else:
        print(f"  Risk Factors: None")

## 6. Comprehensive Volatility Assessment

Perform a complete volatility assessment that combines all the configurable components.

In [ ]:
# Comprehensive volatility assessment
print("=== Comprehensive Volatility Assessment ===")

test_assets = ["ALGO", "USDC", "BTC"]

for asset in test_assets:
    try:
        # Ensure we have some price history
        if asset not in engine.price_history:
            engine.simulate_price_history(asset, hours=24)
        
        # Perform assessment
        metrics = engine.assess_volatility(asset)
        
        print(f"\n{asset} Volatility Assessment:")
        print(f"  7-day Volatility: {metrics.volatility_7d:.4f}")
        print(f"  30-day Volatility: {metrics.volatility_30d:.4f}")
        print(f"  24h Price Range: {metrics.price_range_24h:.4f}")
        print(f"  Price Stability Score: {metrics.price_stability_score:.3f}")
        print(f"  Manipulation Risk: {metrics.manipulation_risk_level}")
        print(f"  Risk Score: {metrics.manipulation_risk_score:.3f}")
        print(f"  Overall Confidence: {metrics.confidence:.3f}")
        print(f"  Oracle Consensus: {metrics.oracle_consensus:.3f}")
        print(f"  Data Freshness: {metrics.data_freshness:.3f}")
        print(f"  Recommendation: {metrics.recommendation}")
        
        if metrics.risk_factors:
            print(f"  Risk Factors: {', '.join(metrics.risk_factors)}")
        
    except Exception as e:
        print(f"\n{asset}: Assessment failed - {e}")

## 7. Configuration Impact Testing

Demonstrate how changing configuration parameters affects the assessment results.

In [ ]:
# Test configuration impact
print("=== Configuration Impact Testing ===")

# Get baseline assessment
baseline_results = engine.detect_price_manipulation("ALGO")
print(f"Baseline Risk Score: {baseline_results['risk_score']:.3f}")
print(f"Baseline Risk Level: {baseline_results['manipulation_risk']}")

# Temporarily modify risk thresholds to be more sensitive
original_volatility_threshold = engine.config.manipulation_detection.risk_thresholds['volatility_high']
original_consensus_threshold = engine.config.manipulation_detection.risk_thresholds['consensus_poor']

print(f"\nOriginal volatility threshold: {original_volatility_threshold}")
print(f"Original consensus threshold: {original_consensus_threshold}")

# Make detection more sensitive
engine.config.manipulation_detection.risk_thresholds['volatility_high'] = 0.05  # 5% instead of 10%
engine.config.manipulation_detection.risk_thresholds['consensus_poor'] = 0.90   # 90% instead of 80%

# Test with more sensitive thresholds
sensitive_results = engine.detect_price_manipulation("ALGO")
print(f"\nSensitive Risk Score: {sensitive_results['risk_score']:.3f}")
print(f"Sensitive Risk Level: {sensitive_results['manipulation_risk']}")

# Restore original thresholds
engine.config.manipulation_detection.risk_thresholds['volatility_high'] = original_volatility_threshold
engine.config.manipulation_detection.risk_thresholds['consensus_poor'] = original_consensus_threshold

print(f"\nConfiguration impact:")
risk_score_change = sensitive_results['risk_score'] - baseline_results['risk_score']
print(f"  Risk score change: {risk_score_change:+.3f}")
print(f"  Risk level change: {baseline_results['manipulation_risk']} → {sensitive_results['manipulation_risk']}")

## 8. Asset Category Analysis

Demonstrate how different asset categories are handled according to configuration.

In [ ]:
# Asset category analysis
print("=== Asset Category Analysis ===")

categories = engine.config.market_analysis.asset_categories

print("Configured Asset Categories:")
for category, assets in categories.items():
    print(f"  {category}: {', '.join(assets)}")

# Test volatility assessment for assets from different categories
test_assets_by_category = {
    'highly_liquid': 'ALGO',
    'liquid': 'AVAX',
    'illiquid': 'GARD'
}

print("\nVolatility by Asset Category:")
for category, asset in test_assets_by_category.items():
    try:
        # Ensure we have price history
        if asset not in engine.price_history:
            engine.simulate_price_history(asset, hours=12)
        
        metrics = engine.assess_volatility(asset)
        
        print(f"\n{category.upper()} ({asset}):")
        print(f"  7d Volatility: {metrics.volatility_7d:.4f}")
        print(f"  Stability Score: {metrics.price_stability_score:.3f}")
        print(f"  Risk Level: {metrics.manipulation_risk_level}")
        
    except Exception as e:
        print(f"\n{category.upper()} ({asset}): Failed - {e}")

## 9. Monitoring Thresholds and Alerts

Demonstrate how the monitoring configuration determines alert levels.

In [ ]:
# Monitoring and alerting demonstration
print("=== Monitoring Configuration ===")

monitoring_config = engine.config.monitoring

print("Alert Thresholds:")
for key, value in monitoring_config.alert_thresholds.items():
    print(f"  {key}: {value}")

print("\nEscalation Levels:")
for key, value in monitoring_config.escalation_levels.items():
    print(f"  {key}: {value}")

# Simulate different risk scores and show escalation
print("\n=== Risk Score Escalation Simulation ===")
test_risk_scores = [0.1, 0.25, 0.4, 0.65, 0.85]

escalation_levels = monitoring_config.escalation_levels

for risk_score in test_risk_scores:
    if risk_score >= escalation_levels['emergency']:
        level = "EMERGENCY"
    elif risk_score >= escalation_levels['critical']:
        level = "CRITICAL"
    elif risk_score >= escalation_levels['warning']:
        level = "WARNING"
    else:
        level = "NORMAL"
    
    print(f"Risk Score {risk_score:.2f} → {level}")

## 10. Summary and Configuration Export

Summarize the configuration system and show how to export current settings.

In [ ]:
# Configuration summary
print("=== Configuration System Summary ===")

print("\n✓ Successfully converted hardcoded parameters to YAML configuration:")
print("  - Price staleness scoring (from _calculate_staleness_score)")
print("  - Volatility calculation windows and thresholds")
print("  - Manipulation detection risk factors and weights")
print("  - Oracle confidence and reliability parameters")
print("  - Market analysis asset categorization")
print("  - Monitoring and alerting thresholds")

print("\n✓ Configuration Features:")
print("  - YAML file-based configuration")
print("  - Environment variable overrides")
print("  - Runtime parameter modification")
print("  - MCP service URL configuration")
print("  - Comprehensive parameter validation")

print("\n✓ Integration Points:")
print(f"  - Algorand Reader MCP: {config.mcp_services.algorand_reader_url}")
print(f"  - Market Data MCP: {config.mcp_services.market_data_url}")

# Show configuration file location
config_path = parent_dir / "config" / "config.yaml"
print(f"\n📁 Configuration file: {config_path}")
print(f"   Exists: {config_path.exists()}")

if config_path.exists():
    file_size = config_path.stat().st_size
    print(f"   Size: {file_size} bytes")

print("\n🎯 Next Steps:")
print("  1. Start MCP services on ports 8002 and 8003")
print("  2. Run: python test_volatility_assessment.py")
print("  3. Run: python test_mcp_integration.py")
print("  4. Customize config/config.yaml for your environment")
print("  5. Set environment variables for production overrides")

In [ ]:
# Final test: validate configuration loading from different sources
print("=== Configuration Loading Validation ===")

try:
    # Test default loading
    config1 = load_config()
    print("✓ Default configuration loading successful")
    
    # Test explicit path loading
    config_file = parent_dir / "config" / "config.yaml"
    if config_file.exists():
        config2 = load_config(config_file)
        print("✓ Explicit path configuration loading successful")
        
        # Verify they're the same
        assert config1.mcp_services.algorand_reader_url == config2.mcp_services.algorand_reader_url
        print("✓ Configuration consistency verified")
    
    # Show sample environment variable usage
    print("\n📋 Environment Variable Examples:")
    print("   export VOLATILITY_MCP_READER_URL=http://localhost:8002")
    print("   export VOLATILITY_MCP_MARKET_URL=http://localhost:8003")
    print("   export VOLATILITY_MIN_CONFIDENCE=0.9")
    print("   export VOLATILITY_CACHE_SECONDS=60")
    print("   export VOLATILITY_DB_PATH=/custom/path/volatility.db")
    
except Exception as e:
    print(f"✗ Configuration validation failed: {e}")

print("\n" + "="*60)
print("🎉 Volatility Assessment Engine Configuration Demo Complete!")
print("="*60)